## Python Bytecode and Performance

### 1️⃣ What is Bytecode?

- Python source code (`.py`) is compiled into **bytecode** (`.pyc`) before execution.
- Bytecode is a **low-level, platform-independent set of instructions** executed by the Python Virtual Machine (PVM).
- It is **not machine code**, but closer to what the CPU executes than raw Python code.

---

### 2️⃣ How Bytecode Relates to Performance

- **Loops and arithmetic** compile to very few instructions → usually fast.
- **Function calls** (like `random.randint()`) generate multiple bytecode instructions → can dominate runtime.
- Profilers (e.g., `cProfile`, `py-spy`) summarize **CPU time per function**, which often correlates with expensive bytecode instructions.


In [1]:
### bytecode example
import random
import dis

def work(n):
    data = []
    for _ in range(n):
        data.append(random.randint(1, 100))
    total = 0
    for x in data:
        total += x * x
    return total

dis.dis(work)

  5           RESUME                   0

  6           BUILD_LIST               0
              STORE_FAST               1 (data)

  7           LOAD_GLOBAL              1 (range + NULL)
              LOAD_FAST                0 (n)
              CALL                     1
              GET_ITER
      L1:     FOR_ITER                40 (to L2)
              STORE_FAST               2 (_)

  8           LOAD_FAST                1 (data)
              LOAD_ATTR                3 (append + NULL|self)
              LOAD_GLOBAL              4 (random)
              LOAD_ATTR                7 (randint + NULL|self)
              LOAD_CONST               1 (1)
              LOAD_CONST               2 (100)
              CALL                     2
              CALL                     1
              POP_TOP
              JUMP_BACKWARD           42 (to L1)

  7   L2:     END_FOR
              POP_TOP

  9           LOAD_CONST               3 (0)
              STORE_FAST               3 (total)


## Bytecode Analysis of `work(n)`

| Line | Instruction              | Argument / Operand         | Meaning / Description |
|------|--------------------------|---------------------------|---------------------|
| 5    | RESUME                   | 0                         | Start/resume function execution |
| 6    | BUILD_LIST               | 0                         | Create empty list `data` |
| 6    | STORE_FAST               | 1 (data)                  | Store the list in local variable `data` |
| 7    | LOAD_GLOBAL              | 1 (range + NULL)          | Load `range` function |
| 7    | LOAD_FAST                | 0 (n)                     | Load local variable `n` |
| 7    | CALL                     | 1                         | Call `range(n)` |
| 7    | GET_ITER                 | -                         | Get iterator from range object |
| 7    | FOR_ITER                 | 40 (to L2)                | Start for-loop iteration |
| 7    | STORE_FAST               | 2 (_)                     | Store loop variable `_` |
| 8    | LOAD_FAST                | 1 (data)                  | Load `data` list |
| 8    | LOAD_ATTR                | 3 (append + NULL|self)    | Prepare `data.append` method |
| 8    | LOAD_GLOBAL              | 4 (random)                | Load `random` module |
| 8    | LOAD_ATTR                | 7 (randint + NULL|self)   | Prepare `random.randint` method |
| 8    | LOAD_CONST               | 1 (1)                     | Load constant `1` |
| 8    | LOAD_CONST               | 2 (100)                   | Load constant `100` |
| 8    | CALL                     | 2                         | Call `randint(1,100)` |
| 8    | CALL                     | 1                         | Call `data.append(value)` |
| 8    | POP_TOP                  | -                         | Remove result of append from stack |
| 8    | JUMP_BACKWARD            | 42 (to L1)                | Jump to start of loop |
| 7 L2 | END_FOR                  | -                         | End of first for loop |
| 7 L2 | POP_TOP                  | -                         | Remove iterator from stack |
| 9    | LOAD_CONST               | 3 (0)                     | Load constant `0` |
| 9    | STORE_FAST               | 3 (total)                 | Initialize `total = 0` |
| 10   | LOAD_FAST                | 1 (data)                  | Load `data` list |
| 10   | GET_ITER                 | -                         | Get iterator for second loop |
| 10   | FOR_ITER                 | 10 (to L4)                | Start second for-loop |
| 10   | STORE_FAST               | 4 (x)                     | Store loop variable `x` |
| 11   | LOAD_FAST_LOAD_FAST      | 52 (total, x)             | Load `total` and `x` |
| 11   | LOAD_FAST                | 4 (x)                     | Load `x` again |
| 11   | BINARY_OP                | 5 (*)                     | Multiply `x * x` |
| 11   | BINARY_OP                | 13 (+=)                   | Add to `total` in place (`total += x*x`) |
| 11   | STORE_FAST               | 3 (total)                 | Store result back in `total` |
| 11   | JUMP_BACKWARD            | 12 (to L3)                | Jump to start of second loop |
| 10 L4 | END_FOR                  | -                         | End of second loop |
| 10 L4 | POP_TOP                  | -                         | Remove iterator from stack |
| 12   | LOAD_FAST                | 3 (total)                 | Load `total` to return |

---

### 🔑 Insights

1. **CPU Hotspots**
   - `CALL` instructions on `random.randint` → dominate CPU usage.
   - `CALL` on `data.append` → minor but repeated 1M times.
2. **Loops are cheap**
   - `FOR_ITER`, `JUMP_BACKWARD`, `END_FOR` → low-cost instructions.
3. **Arithmetic is light**
   - `BINARY_OP (*)` and `BINARY_OP (+=)` → fast compared to function calls.
4. **Profiling matches bytecode**
   - The profiler shows most time in `random.randint()` because **each call generates multiple bytecode instructions** per iteration.
5. **Optimization takeaway**
   - Focus on **reducing expensive function calls** or replacing `random.randint` if performance matters.
6. **operand/arguments**: extra info the VM needs, like variable, constant, or argument count.

## Example
These 2 code snippets below do the same job but the first one generates additional bytecode which cause more overhead

In [2]:
def fn_expressive(upper=1000000):
    total=0
    for n in range(upper):
        total += n
    return total


In [3]:
def fn_terse(upper=1000000):
    return sum(range(upper))

In [7]:
import dis
dis.dis(fn_expressive)

  1           RESUME                   0

  2           LOAD_CONST               1 (0)
              STORE_FAST               1 (total)

  3           LOAD_GLOBAL              1 (range + NULL)
              LOAD_FAST                0 (upper)
              CALL                     1
              GET_ITER
      L1:     FOR_ITER                 7 (to L2)
              STORE_FAST               2 (n)

  4           LOAD_FAST_LOAD_FAST     18 (total, n)
              BINARY_OP               13 (+=)
              STORE_FAST               1 (total)
              JUMP_BACKWARD            9 (to L1)

  3   L2:     END_FOR
              POP_TOP

  5           LOAD_FAST                1 (total)
              RETURN_VALUE


In [8]:
dis.dis(fn_terse)

  1           RESUME                   0

  2           LOAD_GLOBAL              1 (sum + NULL)
              LOAD_GLOBAL              3 (range + NULL)
              LOAD_FAST                0 (upper)
              CALL                     1
              CALL                     1
              RETURN_VALUE
